# Usage Examples

Two end-to-end examples of driving the team with `run_research`, plus a monitor-inspection helper. The runtime agents see only `prompts/` content; this notebook is for human inspection after a run.

1. **Using existing tools** — a question the current `xspec_utils` toolkit can already answer (a joint broadband spectral fit).
2. **Test code authoring** — a question that needs capability the toolkit does *not* expose, so the analyst lead authors the missing tools into `research_toolkit` and the workers rerun on them (Phase 2 step 7).

In [ ]:
from datetime import datetime
import logging
from pathlib import Path
import sys
from IPython.display import Markdown, display
from dotenv import load_dotenv

load_dotenv()

for noisy_logger in ("arxiv", "httpcore", "httpx"):
    logging.getLogger(noisy_logger).setLevel(logging.WARNING)

from auto_research import run_research
from auto_research.monitor import RunMonitor
from auto_research.schemas import ResearchRequest

## Using existing tools

`chatter=7` prints compact live monitor lines while the run is active. The JSONL log still receives every event, including level-10 model-call details. This question is answerable with the toolkit as it stands (`load_data`, `fit_tbfeo_hss`, `run_fit`, `xspec_status`, `plot_fit`, `plot_iqu`, `group_pha`).

In [ ]:
request = ResearchRequest(
    question=(
            "IXPE has performed a Cyg X-1 observation in its soft state on 2023-05-24. "
            "Find simultaneous or near-simultaneous observations from other X-ray observatories "
            "(e.g. NICER, NuSTAR, XMM-Newton) and perform a joint broadband spectral fit "
            "for this epoch. epoch including observations: IXPE: '02008401', NICER: '6643010101' and '6643010102', NuSTAR: '80902318002'."
        ),
        context=(
            "TEST SESSION — evaluation criteria are relaxed as follows:\n"
            "- Absent NuSTAR and NICER backgrounds are acceptable and expected: "
            "Cyg X-1 is an extremely bright source for which background files are not used "
            "(background contribution is negligible; this is standard practice for bright sources).\n"
            "- The primary goal is to verify that analysts correctly invoked the available "
            "toolkit tools (load_data, fit_tbfeo_hss, run_fit, xspec_status, plot_fit, "
            "plot_iqu, group_pha) and that any inline Python executed via "
            "`mcp.builtin.bash_exec` with `python -c` successfully imports xspec and "
            "xspec_utils without errors.\n"
            "- The existing toolkit will not produce publication-level results. "
            "You may send back one round of revision to analysts, asking for better results. "
            "They should attempt to run inline calculations via python -c until they get "
            "non-error outputs and reasonable-looking fit statistics, "
            "but they are not expected to achieve publication-quality fits.\n"
            "- Accept the analysis if tools were called with correct arguments and returned "
            "non-error outputs, even if fit statistics are not publication-quality.\n"
            "- If any of the tools or environments seemed broken, abort the run immediately and report."
        )
)

LOG_DIR = Path("/data/fzhou/auto_research/logs/dev/step-12")
LOG_DIR.mkdir(parents=True, exist_ok=True)
monitor_jsonl = LOG_DIR / f"cyg_x1_ixpe_softstate_{datetime.now().strftime('%Y%m%dT%H%M%S')}_monitor.jsonl"
monitor = RunMonitor(chatter=7, jsonl_path=monitor_jsonl, stream=sys.stdout)
run = await run_research(request, monitor=monitor)
run.monitor.write_markdown(monitor_jsonl.with_suffix(".md"))
monitor_jsonl, monitor_jsonl.with_suffix(".md"), run.verdict

In [ ]:
display(Markdown(run.markdown()))

## Test code authoring

This poses IXPE project **P1 — intra-observation PA/PD stability** (`bhxrb_intraobservation_pa_pd_stability`). Every measurement routes through time-resolved `xpbin algorithm=PCUBE` (PD, PA, Q/I, U/I, MDP99 in ~5 ks segments). The current `xspec_utils` `TOOL_MANIFEST` exposes **only** the spectral-fit workflow — it does not surface PCUBE, the IXPE reduction scripts, or any HEASARC acquire path — so a worker hits the gap and raises a `tool_request`. The analyst lead then authors the missing capability through the gated flow (author → gate → GPT co-lead review → atomic commit on branch `ai`), the registry reloads, sandboxes restart, and the workers rerun on the new tool. With data not local, the realistic chain is **heasarc acquire (new package) → reduction wiring → pcube exposure**, bounded by `tool_author_cap`.

**Prerequisites (human-staged, out of band):** `research_toolkit` (`toolkit_root` in `config/orchestration.yaml`) is on a clean **`ai`** branch with the spectral-only manifest. The launch preflight asserts this and fails cheaply with remediation if not (a dirty tree usually means a prior terminal authoring failure left edits to inspect). A reachable `human_handler`/channel is recommended so the lead/author can escalate; an honest `failed`/`needs_human` outcome is a *correct* result — never fabricated.

In [ ]:
authoring_request = ResearchRequest(
    question=(
        "Quantify the intra-observation stability of polarization angle (PA) and polarization "
        "degree (PD) in long-exposure BH-XRB IXPE observations using time-resolved PCUBE "
        "analysis. For each observation: define a 60-arcsec source region, merge DU1/DU2/DU3, "
        "filter to 2-8 keV, divide the GTI exposure into ~5 ks segments, run xpbin "
        "algorithm=PCUBE (NEFF weighting) per segment to extract PD, PA, Q/I, U/I and MDP99, "
        "then apply weighted chi-squared constancy tests (dof = N_segments - 1, p-values via "
        "scipy.stats.chi2.sf) to flag significant PA wander or PD variability. Observations: "
        "Cyg X-1 (01002901, 03002599), Swift J1727.8-1613 (03005801, 03006001), "
        "4U 1630-47 (01250401), LMC X-1 (02001901)."
    ),
    context=(
        "TEST SESSION — this exercises Phase 2 step 7 (lead-driven tool authoring). The current "
        "xspec_utils toolkit exposes only the spectral-fit workflow; it does NOT expose a PCUBE "
        "polarimetry tool, the IXPE reduction scripts, or a HEASARC data-acquisition path. When "
        "a worker finds the capability missing, it should raise a tool_request (never fabricate "
        "numbers); the lead browses research_toolkit/lib read-only, decides extend-vs-new, and "
        "authors the missing tool(s) in dependency order (acquire -> reduce -> pcube). Each "
        "authored tool is gated and reviewed before an atomic commit on branch 'ai'; the workers "
        "then rerun on it. Evaluation: verify the authoring loop runs end-to-end and the analysis "
        "completes on a freshly authored tool. An honest failed/needs_human outcome (e.g. a tool "
        "that cannot be made mergeable within the rework caps) is an acceptable, correct result — "
        "do NOT fabricate a polarization measurement. If the environment is broken, abort and report."
    ),
)

AUTHORING_LOG_DIR = Path("/data/fzhou/auto_research/logs/dev/step-7")
AUTHORING_LOG_DIR.mkdir(parents=True, exist_ok=True)
authoring_jsonl = AUTHORING_LOG_DIR / f"p1_pa_pd_stability_{datetime.now().strftime('%Y%m%dT%H%M%S')}_monitor.jsonl"
authoring_monitor = RunMonitor(chatter=7, jsonl_path=authoring_jsonl, stream=sys.stdout)
# human_channel='cli' lets the lead/author escalate to you; use 'whatsapp'/'slack' for a remote run.
authoring_run = await run_research(authoring_request, monitor=authoring_monitor, human_channel="cli")
authoring_run.monitor.write_markdown(authoring_jsonl.with_suffix(".md"))
authoring_jsonl, authoring_jsonl.with_suffix(".md"), authoring_run.verdict

In [ ]:
display(Markdown(authoring_run.markdown()))

## Inspect An Existing Monitor

In [ ]:
# Inspect the most recent monitor log. Override `log_dir`/`existing_jsonl` to point elsewhere
# (e.g. .../logs/dev/step-7 for an authoring run).
log_dir = Path("/data/fzhou/auto_research/logs/dev/step-12")
candidates = sorted(log_dir.glob("*.jsonl")) if log_dir.exists() else []
existing_jsonl = candidates[-1] if candidates else None

if existing_jsonl:
    rows = [l for l in existing_jsonl.read_text(encoding="utf-8").splitlines() if l.strip()]
    print(f"{len(rows)} monitor events in {existing_jsonl}")
    md = existing_jsonl.with_suffix(".md")
    if md.exists():
        display(Markdown(md.read_text(encoding="utf-8")))
else:
    print(f"No logs yet in {log_dir}")